In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%%capture
!pip install torch-geometric

In [3]:
import torch
import torch.nn.functional as F
# print(f"GPU: {torch.cuda.get_device_name(0)}")
# print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

In [4]:
import os
DATA_DIR = "/content/drive/MyDrive/Data_Amazon"

files = ["train.parquet", "val.parquet", "test.parquet", "id_mappings.json"]


Load Data

In [5]:
import json
import pandas as pd
import numpy as np

train_df = pd.read_parquet(f"{DATA_DIR}/train.parquet")
val_df   = pd.read_parquet(f"{DATA_DIR}/val.parquet")
test_df  = pd.read_parquet(f"{DATA_DIR}/test.parquet")

with open(f"{DATA_DIR}/id_mappings.json") as f:
    mappings = json.load(f)

n_users  = mappings["n_users"]
n_items  = mappings["n_items"]
user2idx = mappings["user2idx"]
item2idx = mappings["item2idx"]

print(f"train : {len(train_df):,}")
print(f"val   : {len(val_df):,}")
print(f"test  : {len(test_df):,}")
print(f"n_users = {n_users:,}  |  n_items = {n_items:,}")

train : 8,260,845
val   : 1,143,887
test  : 1,141,932
n_users = 1,145,516  |  n_items = 283,184


In [6]:
item_counts = train_df["item_idx"].value_counts()
all_items   = pd.Series(0, index=range(n_items))
all_items.update(item_counts)


p80 = all_items.quantile(0.80)

item_group = all_items.apply(lambda c: "head" if c > p80 else "tail")
item_group_dict = item_group.to_dict()

print(f"head: {(item_group=='head').sum():,} items ({p80:.0f} interactions)")
print(f"tail: {(item_group=='tail').sum():,} items (<={p80:.0f} interactions)")

head: 55,030 items (25 interactions)
tail: 228,154 items (<=25 interactions)


In [7]:
num_users_train = train_df["user_idx"].nunique()
num_items_train = len(all_items)

total_train_interactions = len(train_df)
max_possible_train_interactions = num_users_train * num_items_train

train_sparsity = (total_train_interactions / max_possible_train_interactions)
print(f"Density: {train_sparsity:.7f}")

Density: 0.0000255


Model

In [8]:
from torch_geometric.nn import LGConv
import torch
import torch.nn as nn

class LightGCN(nn.Module):
    def __init__(self, n_users, n_items, embedding_dim=64, n_layers=3):
        super(LightGCN, self).__init__()
        self.n_users = n_users
        self.n_items = n_items
        self.embedding_dim = embedding_dim
        self.n_layers = n_layers

        self.convs = nn.ModuleList([LGConv() for _ in range(n_layers)])

        self.user_emb = nn.Embedding(n_users, embedding_dim)
        self.item_emb = nn.Embedding(n_items, embedding_dim)

        nn.init.xavier_uniform_(self.user_emb.weight)
        nn.init.xavier_uniform_(self.item_emb.weight)

    def build_graph(self, train_df, device):
        user_ids = torch.tensor(train_df['user_idx'].values, dtype=torch.long)
        item_ids = torch.tensor(train_df['item_idx'].values, dtype=torch.long) + self.n_users

        edge_index_user_to_item = torch.stack([user_ids, item_ids], dim=0)
        edge_index_item_to_user = torch.stack([item_ids, user_ids], dim=0)

        self.edge_index = torch.cat([edge_index_user_to_item, edge_index_item_to_user], dim=1).to(device)

    def forward(self):
        x = torch.cat([self.user_emb.weight, self.item_emb.weight], dim=0)
        layers = [x]
        for conv in self.convs:
            x = conv(x, self.edge_index)
            layers.append(x)
        final = torch.stack(layers, dim=1).mean(dim=1)
        return final[:self.n_users], final[self.n_users:]

    def bpr_loss(self, users, pos_items, neg_items, final_user_emb, final_item_emb):
        u   = final_user_emb[users]
        pos = final_item_emb[pos_items]
        neg = final_item_emb[neg_items]

        pos_score = (u * pos).sum(-1)
        neg_score = (u * neg).sum(-1)

        loss = -torch.log(torch.sigmoid(pos_score - neg_score) + 1e-8).mean()

        reg = (self.user_emb.weight[users].norm(2)**2 +
               self.item_emb.weight[pos_items].norm(2)**2 +
               self.item_emb.weight[neg_items].norm(2)**2) / len(users)
        return loss, reg

BPR Dataset

In [9]:
import numpy as np
from torch.utils.data import Dataset, DataLoader

class BPRDataset(Dataset):
    def __init__(self, train_df, n_items):
        self.n_items  = n_items
        self.pairs    = train_df[["user_idx", "item_idx"]].values
        self.user_pos = train_df.groupby("user_idx")["item_idx"].apply(set).to_dict()

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        u, pos = int(self.pairs[idx, 0]), int(self.pairs[idx, 1])
        while True:
            neg = np.random.randint(0, self.n_items)
            if neg not in self.user_pos[u]:
                break
        return torch.tensor(u), torch.tensor(pos), torch.tensor(neg)

dataset = BPRDataset(train_df, n_items)
u, p, n = dataset[1]
print(f"Sample: user={u.item()}, pos={p.item()}, neg={n.item()}")
print(f"Dataset size: {len(dataset):,}")

Sample: user=0, pos=12346, neg=197398
Dataset size: 8,260,845


Evaluation

In [10]:
@torch.no_grad()
def evaluate(model, eval_df, train_df, topks, device, item_group, batch_size=1024):
    model.eval()
    user_emb, item_emb = model.forward()
    train_history = train_df.groupby("user_idx")["item_idx"].apply(set).to_dict()

    users   = eval_df["user_idx"].values
    targets = eval_df["item_idx"].values

    results = {k: {"head": [], "tail": [], "all": []} for k in topks}

    for start in range(0, len(users), batch_size):
        batch_users   = users[start: start + batch_size]
        batch_targets = targets[start: start + batch_size]

        u_t    = torch.tensor(batch_users, dtype=torch.long, device=device)
        u_emb  = user_emb[u_t]
        scores = u_emb @ item_emb.T

        for i, u in enumerate(batch_users):
            train_pos = list(train_history.get(int(u), set()))
            if train_pos:
                scores[i, train_pos] = -1e9

        target_t      = torch.tensor(batch_targets, dtype=torch.long, device=device)
        target_scores = scores[torch.arange(len(batch_users), device=device), target_t]
        ranks         = (scores > target_scores.unsqueeze(1)).sum(dim=1) + 1

        for i, (u, t) in enumerate(zip(batch_users, batch_targets)):
            rank = ranks[i].item()
            g    = item_group_dict.get(int(t), "tail")
            for k in topks:
                hit  = 1.0 if rank <= k else 0.0
                ndcg = (1.0 / np.log2(rank + 1)) if rank <= k else 0.0
                results[k][g].append((hit, ndcg))
                results[k]["all"].append((hit, ndcg))

    model.train()

    out = {}
    for k in topks:
        out[k] = {}
        for g, vals in results[k].items():
            if vals:
                hits, ndcgs = zip(*vals)
                out[k][g] = {"recall": np.mean(hits), "ndcg": np.mean(ndcgs), "n": len(vals)}
    return out

Config

In [11]:
DEVICE      = torch.device("cuda")
EMB_DIM     = 64
N_LAYERS    = 3
LR          = 4e-3
N_EPOCHS    = 20
BATCH_SIZE  = 8192
PATIENCE    = 10
TOPKS       = [20, 40]
EVAL_EVERY  = 2

Init Model

In [12]:
import torch.optim as optim
from torch_geometric.data import Data

model = LightGCN(n_users, n_items, EMB_DIM, N_LAYERS).to(DEVICE)
model.build_graph(train_df, DEVICE)
loader    = DataLoader(BPRDataset(train_df, n_items),
                       batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=0)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS)

print(f"Model params: {sum(p.numel() for p in model.parameters()):,}")

Model params: 91,436,800


Training loop

In [13]:
from tqdm import tqdm
import time

best_recall = 0.0
best_epoch  = 0
no_improve  = 0
OUTPUT_DIR  = "/content/lightgcn_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

for epoch in range(1, N_EPOCHS + 1):
    model.train()
    total_loss = 0.0
    t0 = time.time()

    pbar = tqdm(loader, desc=f"Epoch {epoch:3d}/{N_EPOCHS}", leave=False, dynamic_ncols=True)

    for users, pos_items, neg_items in pbar:
        users, pos_items, neg_items = (
            users.to(DEVICE), pos_items.to(DEVICE), neg_items.to(DEVICE)
        )
        optimizer.zero_grad()

        final_user_emb, final_item_emb = model.forward()

        loss, reg = model.bpr_loss(users, pos_items, neg_items, final_user_emb, final_item_emb)

        (loss + 1e-4 * reg).backward()

        optimizer.step()

        total_loss += loss.item()
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    avg_loss = total_loss / len(loader)

    scheduler.step()

    if epoch % EVAL_EVERY == 0 or epoch == N_EPOCHS:
        metrics = evaluate(model, val_df, train_df, TOPKS, DEVICE, item_group_dict)
        recall  = metrics[20]["all"]["recall"]
        improved = recall > best_recall

        if improved:
            best_recall = recall
            best_epoch  = epoch
            no_improve  = 0
            torch.save(model.state_dict(), f"{OUTPUT_DIR}/best_model.pt")
        else:
            no_improve += EVAL_EVERY

        print(f"Epoch {epoch:3d} | loss={avg_loss:.4f} | {time.time()-t0:.1f}")

        for k in TOPKS:
            print(f"  --- @{k} ---")
            for g in ["head", "tail", "all"]:
                m = metrics[k][g]
                print(f"  {g:>4} (n={m['n']:>6,}) | Recall@{k}={m['recall']:.4f} | NDCG@{k}={m['ndcg']:.4f}")

        if no_improve >= PATIENCE:
            print(f"\nEarly stopping! Best Recall@{TOPK}={best_recall:.4f} @ epoch {best_epoch}")
            break
    else:
        print(f"Epoch {epoch:3d} | loss={avg_loss:.4f} | {time.time()-t0:.1f}s")

    current_lr = optimizer.param_groups[0]['lr']
    if(epoch != N_EPOCHS):
        print(f"LR={current_lr:.2e}")


Epoch   1 | loss=0.3480 | 653.6s
LR=3.98e-03


Epoch   2 | loss=0.2706 | 796.6
  --- @20 ---
  head (n=826,199) | Recall@20=0.0415 | NDCG@20=0.0177
  tail (n=317,688) | Recall@20=0.0001 | NDCG@20=0.0000
   all (n=1,143,887) | Recall@20=0.0300 | NDCG@20=0.0128
  --- @40 ---
  head (n=826,199) | Recall@40=0.0608 | NDCG@40=0.0216
  tail (n=317,688) | Recall@40=0.0001 | NDCG@40=0.0000
   all (n=1,143,887) | Recall@40=0.0440 | NDCG@40=0.0156
LR=3.90e-03


Epoch   3 | loss=0.2185 | 652.5s
LR=3.78e-03


Epoch   4 | loss=0.1768 | 795.9
  --- @20 ---
  head (n=826,199) | Recall@20=0.0467 | NDCG@20=0.0198
  tail (n=317,688) | Recall@20=0.0002 | NDCG@20=0.0001
   all (n=1,143,887) | Recall@20=0.0338 | NDCG@20=0.0144
  --- @40 ---
  head (n=826,199) | Recall@40=0.0689 | NDCG@40=0.0244
  tail (n=317,688) | Recall@40=0.0003 | NDCG@40=0.0001
   all (n=1,143,887) | Recall@40=0.0499 | NDCG@40=0.0176
LR=3.62e-03


Epoch   5 | loss=0.1516 | 652.5s
LR=3.41e-03


...

Epoch  16 | loss=0.0590 | 797.9
  --- @20 ---
  head (n=826,199) | Recall@20=0.0522 | NDCG@20=0.0221
  tail (n=317,688) | Recall@20=0.0005 | NDCG@20=0.0002
   all (n=1,143,887) | Recall@20=0.0378 | NDCG@20=0.0160
  --- @40 ---
  head (n=826,199) | Recall@40=0.0762 | NDCG@40=0.0270
  tail (n=317,688) | Recall@40=0.0008 | NDCG@40=0.0002
   all (n=1,143,887) | Recall@40=0.0553 | NDCG@40=0.0196
LR=3.82e-04


Epoch  17 | loss=0.0574 | 653.3s
LR=2.18e-04


Epoch  18 | loss=0.0568 | 799.4
  --- @20 ---
  head (n=826,199) | Recall@20=0.0523 | NDCG@20=0.0222
  tail (n=317,688) | Recall@20=0.0005 | NDCG@20=0.0002
   all (n=1,143,887) | Recall@20=0.0379 | NDCG@20=0.0161
  --- @40 ---
  head (n=826,199) | Recall@40=0.0765 | NDCG@40=0.0271
  tail (n=317,688) | Recall@40=0.0008 | NDCG@40=0.0002
   all (n=1,143,887) | Recall@40=0.0555 | NDCG@40=0.0196
LR=9.79e-05


Epoch  19 | loss=0.0561 | 652.7s
LR=2.46e-05


Epoch  20 | loss=0.0559 | 794.9
  --- @20 ---
  head (n=826,199) | Recall@20=0.0524 | NDCG@20=0.0222
  tail (n=317,688) | Recall@20=0.0005 | NDCG@20=0.0002
   all (n=1,143,887) | Recall@20=0.0380 | NDCG@20=0.0161
  --- @40 ---
  head (n=826,199) | Recall@40=0.0765 | NDCG@40=0.0271
  tail (n=317,688) | Recall@40=0.0008 | NDCG@40=0.0002
   all (n=1,143,887) | Recall@40=0.0555 | NDCG@40=0.0196


Test model

In [15]:
model.load_state_dict(torch.load(f"{OUTPUT_DIR}/best_model.pt"))

metrics = evaluate(model, test_df, train_df, TOPKS, DEVICE, item_group_dict)

for k in TOPKS:
    print(f"  --- @{k} ---")
    for g in ["head", "tail", "all"]:
        m = metrics[k][g]
        print(f"  {g:>4} (n={m['n']:>6,}) | Recall@{k}={m['recall']:.4f} | NDCG@{k}={m['ndcg']:.4f}")

  --- @20 ---
  head (n=782,213) | Recall@20=0.0381 | NDCG@20=0.0158
  tail (n=359,719) | Recall@20=0.0003 | NDCG@20=0.0001
   all (n=1,141,932) | Recall@20=0.0262 | NDCG@20=0.0108
  --- @40 ---
  head (n=782,213) | Recall@40=0.0568 | NDCG@40=0.0196
  tail (n=359,719) | Recall@40=0.0005 | NDCG@40=0.0002
   all (n=1,141,932) | Recall@40=0.0391 | NDCG@40=0.0135


In [16]:
import shutil
shutil.copy(f"{OUTPUT_DIR}/best_model.pt",
            f"{DATA_DIR}/lightgcn_best.pt")
print("Saved to Drive")

Saved to Drive
